In [0]:
schema = 'fifa_bi_dev.gold_schema'

In [0]:
from pyspark.sql.functions import *

matches = spark.read.table('fifa_bi_dev.silver_schema.silver_world_cup_matches')
temperature = spark.read.table('fifa_bi_dev.silver_schema.silver_monthly_temperature')
temperature_adj = spark.read.table('fifa_bi_dev.silver_schema.seed_temp_adjustments')

m = matches.alias('m')
t = temperature.alias('t')
ta = temperature_adj.alias('ta')

result = m.join(t, (col('m.month') == col('t.month')) & (col('m.host_country') == col('t.country')), 'left').drop('month','monthly_average', 'country','country_code')
result = result.join(ta, (hour(col('time')) >= col('ta.start_hour')) & (hour(col('time')) < col('ta.end_hour')), 'left').drop('time_period','start_hour', 'end_hour')


final = result.select('tournament_name',
              'year', 
              'match_id', 
              'fixture_id',
              'team_1_code',
              'team_1_country',
              'team_2_code',
              'team_2_country', 
              'team_1_goals',
              'team_2_goals',
              'team_1_penalties',
              'team_2_penalties',
              'winner_country',
              'match_finish_type',
              (col('avg_temperature') + col('temperature_adjustment_c')).alias('adj_temperature_final')
              ).filter(col('year')>= '1940')

final.show()


In [0]:
final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f'{schema}.gold_match_temperature')
print(f'table_name: gold_match_temperature is updated')